# 14. 유저 반응 확보 × 초기 만족도 2x2 포지셔닝

**분석 목적:** 같은 메타데이터 속성이라도 어떤 속성은 `리뷰 10~49개` 수준의 첫 유저 반응을 잘 만들고, 어떤 속성은 반응 이후에도 높은 만족도까지 이어진다. 이 노트북은 속성별로 `유저 반응 확보율`과 `초기 만족도 high 비율`을 함께 비교해 출시 전략상 위치를 파악한다.

**해석 프레임:**
- x축: `유저 반응 확보율` = 해당 속성의 전체 게임 중 리뷰 `10~49개`에 도달한 비율

  ```text
  유저 반응 확보율
  = 유저 반응 그룹 게임 수 / (침묵 그룹 게임 수 + 유저 반응 그룹 게임 수)
  ```

- y축: `초기 만족도 high 비율` = 유저 반응을 확보한 게임 중 긍정률 `80% 이상` 비율
- 우상단: 반응도 잘 얻고 만족도도 높은 속성
- 우하단: 반응은 얻지만 만족도가 불안정한 속성
- 좌상단: 반응은 적지만 반응한 뒤 만족도는 높은 속성
- 좌하단: 반응 확보와 만족도 모두 약한 속성

In [11]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px


## 1. 데이터 로드 및 분석 기준 정의

`steam_indie_games_silence.csv`에서 침묵 그룹(리뷰 0~9개), `steam_indie_games_graded.csv`에서 유저 반응 그룹(리뷰 10~49개)만 불러와 결합한다. 만족도는 유저 반응 그룹 내부에서만 계산한다.

In [12]:
DATA_DIR = Path('../../../data/preprocessed')
SILENCE_PATH = DATA_DIR / 'steam_indie_games_silence.csv'
GRADED_PATH = DATA_DIR / 'steam_indie_games_graded.csv'

silence = pd.read_csv(SILENCE_PATH)
response_all = pd.read_csv(GRADED_PATH)
response = response_all[response_all['scale_grade'] == 'low'].copy()

TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']
TARGET_CATEGORIES = ['Single-player', 'Steam Achievements', 'Full controller support', 'Online Co-op', 'PvP']


def parse_list_column(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []


def parse_csv_like_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        return [item.strip() for item in value.split(',') if item.strip()]
    return []


def parse_tags(value):
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return list(parsed.keys())
            if isinstance(parsed, list):
                return parsed
        except Exception:
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, dict):
                    return list(parsed.keys())
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return []
    return []

for df in [silence, response]:
    df['genres'] = df['genres'].apply(parse_list_column)
    df['category_list'] = df['categories'].apply(parse_csv_like_list)
    df['tag_list'] = df['tags'].apply(parse_tags) if 'tags' in df.columns else [[] for _ in range(len(df))]
    df['description_length'] = df['short_description'].fillna('').str.len()

response['is_high_satisfaction'] = response['satisfaction_grade'].eq('high')
response['group_label'] = '유저 반응 (리뷰 10~49개)'
silence['group_label'] = '침묵 (리뷰 0~9개)'

overall_response_capture_rate = len(response) / (len(response) + len(silence)) * 100
overall_high_satisfaction_rate = response['is_high_satisfaction'].mean() * 100

print(f'침묵 그룹: {len(silence):,}개')
print(f'유저 반응 그룹: {len(response):,}개')
print(f'전체: {len(silence) + len(response):,}개')
print(f'전체 평균 유저 반응 확보율: {overall_response_capture_rate:.1f}%')
print(f'전체 평균 초기 만족도 high 비율: {overall_high_satisfaction_rate:.1f}%')

침묵 그룹: 6,737개
유저 반응 그룹: 4,904개
전체: 11,641개
전체 평균 유저 반응 확보율: 42.1%
전체 평균 초기 만족도 high 비율: 71.7%


**해석:** 이 분석의 기준선은 전체 평균 `유저 반응 확보율`과 `초기 만족도 high 비율`이다. 각 속성은 이 평균선보다 위/아래 어디에 있는지로 상대적 강점과 약점을 해석한다.

## 2. 장르별 2x2 포지셔닝

장르별로 침묵 게임과 유저 반응 게임을 합쳐 `유저 반응 확보율`을 계산하고, 유저 반응을 얻은 장르 안에서 `초기 만족도 high 비율`을 계산한다.

In [13]:
def build_attribute_matrix(all_df, response_df, attribute_name, target_values):
    rows = []
    for value in target_values:
        total_games = all_df[attribute_name].apply(lambda items: value in items).sum()
        response_games = response_df[attribute_name].apply(lambda items: value in items).sum()
        if total_games == 0 or response_games == 0:
            continue
        high_games = response_df[response_df[attribute_name].apply(lambda items: value in items)]['is_high_satisfaction'].sum()
        rows.append({
            'attribute': value,
            'total_games': int(total_games),
            'response_games': int(response_games),
            'high_satisfaction_games': int(high_games),
            '유저 반응 확보율(%)': round(response_games / total_games * 100, 1),
            '초기 만족도 high 비율(%)': round(high_games / response_games * 100, 1),
        })
    return pd.DataFrame(rows)


genre_all = pd.concat([
    silence[['appid', 'genres']].copy(),
    response[['appid', 'genres']].copy(),
], ignore_index=True)

genre_matrix = build_attribute_matrix(genre_all, response[['appid', 'genres', 'is_high_satisfaction']], 'genres', TARGET_GENRES)
genre_matrix = genre_matrix.sort_values(['유저 반응 확보율(%)', '초기 만족도 high 비율(%)'], ascending=[False, False]).reset_index(drop=True)
display(genre_matrix)

fig = px.scatter(
    genre_matrix,
    x='유저 반응 확보율(%)',
    y='초기 만족도 high 비율(%)',
    size='total_games',
    text='attribute',
    color='초기 만족도 high 비율(%)',
    color_continuous_scale='Blues',
    hover_data={'total_games': True, 'response_games': True, 'high_satisfaction_games': True},
    title='장르별 유저 반응 확보율 × 초기 만족도 high 비율',
    labels={'attribute': '장르'}
)
fig.add_vline(x=overall_response_capture_rate, line_dash='dash', line_color='gray')
fig.add_hline(y=overall_high_satisfaction_rate, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=900, height=600, coloraxis_showscale=False)
fig.show()

,attribute,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
0,Simulation,2359,1115,626,47.3,56.1
1,Adventure,5356,2472,1689,46.2,68.3
2,RPG,2057,904,600,43.9,66.4
3,Sports,438,189,123,43.2,65.1
4,Action,5282,2214,1580,41.9,71.4
5,Strategy,2366,945,670,39.9,70.9
6,Racing,454,178,109,39.2,61.2
7,Casual,5955,2330,1721,39.1,73.9


**해석 가이드:** 우상단 장르는 출시 직후 반응도 잘 얻고 만족도도 안정적인 장르다. 반대로 좌하단 장르는 첫 반응 확보와 만족도 둘 다 어려운 편이므로, 같은 장르를 선택하더라도 가격·태그·설명 방식까지 더 보수적으로 설계할 필요가 있다.

## 3. 가격대별 2x2 포지셔닝

가격대는 구매 장벽과 기대 품질을 동시에 반영한다. 각 가격대에서 리뷰 10~49개 반응을 얻을 확률과, 반응 이후 긍정률 80% 이상을 유지할 확률을 함께 본다.

In [14]:
PRICE_BINS = [0, 5, 10, 15, 20, 30, 60, float('inf')]
PRICE_LABELS = ['~$5', '$5~10', '$10~15', '$15~20', '$20~30', '$30~60', '$60+']

silence_price = silence[silence['price'] > 0].copy()
response_price = response[response['price'] > 0].copy()
for df in [silence_price, response_price]:
    df['price_range'] = pd.cut(df['price'], bins=PRICE_BINS, labels=PRICE_LABELS, right=False, include_lowest=True)

price_all = pd.concat([
    silence_price[['appid', 'price_range']].copy(),
    response_price[['appid', 'price_range']].copy(),
], ignore_index=True)

price_rows = []
for value in PRICE_LABELS:
    total_games = int((price_all['price_range'] == value).sum())
    response_games = int((response_price['price_range'] == value).sum())
    if total_games == 0 or response_games == 0:
        continue
    high_games = int(response_price.loc[response_price['price_range'] == value, 'is_high_satisfaction'].sum())
    price_rows.append({
        'attribute': value,
        'total_games': total_games,
        'response_games': response_games,
        'high_satisfaction_games': high_games,
        '유저 반응 확보율(%)': round(response_games / total_games * 100, 1),
        '초기 만족도 high 비율(%)': round(high_games / response_games * 100, 1),
    })
price_matrix = pd.DataFrame(price_rows)
display(price_matrix)

fig = px.scatter(
    price_matrix,
    x='유저 반응 확보율(%)',
    y='초기 만족도 high 비율(%)',
    size='total_games',
    text='attribute',
    color='유저 반응 확보율(%)',
    color_continuous_scale='Teal',
    hover_data={'total_games': True, 'response_games': True, 'high_satisfaction_games': True},
    title='가격대별 유저 반응 확보율 × 초기 만족도 high 비율',
    labels={'attribute': '가격대'}
)
fig.add_vline(x=overall_response_capture_rate, line_dash='dash', line_color='gray')
fig.add_hline(y=overall_high_satisfaction_rate, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=900, height=600, coloraxis_showscale=False)
fig.show()

,attribute,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
0,~$5,7467,2751,1969,36.8,71.6
1,$5~10,2816,1378,1007,48.9,73.1
2,$10~15,835,483,368,57.8,76.2
3,$15~20,323,191,123,59.1,64.4
4,$20~30,123,67,35,54.5,52.2
5,$30~60,42,15,9,35.7,60.0
6,$60+,35,19,7,54.3,36.8


**해석 가이드:** 오른쪽으로 갈수록 해당 가격대가 유저 반응을 확보하기 쉽고, 위로 갈수록 그 반응이 높은 만족도로 이어질 확률이 높다. 특히 고가 구간은 표본 수가 적을 수 있으므로, 위치와 함께 `total_games`도 반드시 같이 봐야 한다.

## 4. 플랫폼·카테고리별 2x2 포지셔닝

유저가 상점 페이지에서 바로 확인하는 메타데이터도 반응과 만족도에 영향을 준다. 여기서는 플랫폼 지원과 주요 카테고리 몇 가지를 같은 방식으로 비교한다.

In [15]:
platform_rows = []
for label, col in [('Windows 지원', 'windows'), ('Mac 지원', 'mac'), ('Linux 지원', 'linux')]:
    total_games = int(pd.concat([silence[col], response[col]], ignore_index=True).fillna(False).astype(bool).sum())
    response_games = int(response[col].fillna(False).astype(bool).sum())
    if total_games == 0 or response_games == 0:
        continue
    high_games = int(response.loc[response[col].fillna(False).astype(bool), 'is_high_satisfaction'].sum())
    platform_rows.append({
        'attribute': label,
        'total_games': total_games,
        'response_games': response_games,
        'high_satisfaction_games': high_games,
        '유저 반응 확보율(%)': round(response_games / total_games * 100, 1),
        '초기 만족도 high 비율(%)': round(high_games / response_games * 100, 1),
    })
platform_matrix = pd.DataFrame(platform_rows)

def build_category_matrix(all_df, response_df, categories):
    rows = []
    for cat in categories:
        total_games = all_df['category_list'].apply(lambda items: cat in items).sum()
        response_games = response_df['category_list'].apply(lambda items: cat in items).sum()
        if total_games == 0 or response_games == 0:
            continue
        high_games = response_df.loc[response_df['category_list'].apply(lambda items: cat in items), 'is_high_satisfaction'].sum()
        rows.append({
            'attribute': cat,
            'total_games': int(total_games),
            'response_games': int(response_games),
            'high_satisfaction_games': int(high_games),
            '유저 반응 확보율(%)': round(response_games / total_games * 100, 1),
            '초기 만족도 high 비율(%)': round(high_games / response_games * 100, 1),
        })
    return pd.DataFrame(rows)

category_all = pd.concat([silence[['appid', 'category_list']], response[['appid', 'category_list']]], ignore_index=True)
category_matrix = build_category_matrix(category_all, response[['appid', 'category_list', 'is_high_satisfaction']], TARGET_CATEGORIES)
meta_matrix = pd.concat([platform_matrix, category_matrix], ignore_index=True)
display(meta_matrix)

fig = px.scatter(
    meta_matrix,
    x='유저 반응 확보율(%)',
    y='초기 만족도 high 비율(%)',
    size='total_games',
    text='attribute',
    color='초기 만족도 high 비율(%)',
    color_continuous_scale='Viridis',
    hover_data={'total_games': True, 'response_games': True, 'high_satisfaction_games': True},
    title='플랫폼·카테고리별 유저 반응 확보율 × 초기 만족도 high 비율',
    labels={'attribute': '메타데이터 속성'}
)
fig.add_vline(x=overall_response_capture_rate, line_dash='dash', line_color='gray')
fig.add_hline(y=overall_high_satisfaction_rate, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=1000, height=650, coloraxis_showscale=False)
fig.show()

,attribute,total_games,response_games,high_satisfaction_games,유저 반응 확보율(%),초기 만족도 high 비율(%)
0,Windows 지원,11564,4883,3506,42.2,71.8
1,Mac 지원,1575,788,636,50.0,80.7
2,Linux 지원,1392,667,552,47.9,82.8
3,Single-player,11363,4803,3438,42.3,71.6
4,Steam Achievements,6114,3020,2324,49.4,77.0
5,Full controller support,3084,1590,1310,51.6,82.4
6,Online Co-op,420,209,132,49.8,63.2
7,PvP,788,304,242,38.6,79.6


**해석 가이드:** 이 그래프는 장르나 가격보다 더 실행 가능한 메타데이터 신호를 보여준다. 예를 들어 어떤 카테고리가 오른쪽에 있다면 유저 반응 확보에는 유리했고, 위쪽에 있다면 반응 이후 만족도 관리에도 도움이 됐다는 뜻이다.

## 5. 태그별 2x2 포지셔닝

태그가 침묵 그룹까지 확보되었으므로, 이제 태그별로도 `유저 반응 확보율`과 `초기 만족도 high 비율`을 함께 비교할 수 있다. 태그는 유저가 기대하는 경험의 구체적 내용을 가장 직접적으로 반영하는 속성이다.

In [ ]:
MIN_TAG_TOTAL = 80
MIN_TAG_RESPONSE = 20
GENERAL_TAGS = set(TARGET_GENRES + ['Indie', 'Singleplayer', 'Multiplayer', '2D', '3D'])

def build_tag_matrix(all_df, response_df):
    all_tags = all_df[['appid', 'tag_list']].explode('tag_list').dropna(subset=['tag_list']).copy()
    response_tags = response_df[['appid', 'tag_list', 'is_high_satisfaction']].explode('tag_list').dropna(subset=['tag_list']).copy()
    all_tags = all_tags[all_tags['tag_list'].astype(str).str.len() > 0]
    response_tags = response_tags[response_tags['tag_list'].astype(str).str.len() > 0]

    total_counts = all_tags.groupby('tag_list')['appid'].count().rename('total_games')
    response_counts = response_tags.groupby('tag_list')['appid'].count().rename('response_games')
    high_counts = response_tags.groupby('tag_list')['is_high_satisfaction'].sum().rename('high_satisfaction_games')

    tag_matrix = pd.concat([total_counts, response_counts, high_counts], axis=1).fillna(0).reset_index().rename(columns={'tag_list': 'attribute'})
    tag_matrix[['total_games', 'response_games', 'high_satisfaction_games']] = tag_matrix[['total_games', 'response_games', 'high_satisfaction_games']].astype(int)
    tag_matrix = tag_matrix[(tag_matrix['total_games'] >= MIN_TAG_TOTAL) & (tag_matrix['response_games'] >= MIN_TAG_RESPONSE)]
    tag_matrix = tag_matrix[~tag_matrix['attribute'].isin(GENERAL_TAGS)].copy()
    tag_matrix['유저 반응 확보율(%)'] = (tag_matrix['response_games'] / tag_matrix['total_games'] * 100).round(1)
    tag_matrix['초기 만족도 high 비율(%)'] = (tag_matrix['high_satisfaction_games'] / tag_matrix['response_games'] * 100).round(1)
    return tag_matrix.sort_values(['total_games', '유저 반응 확보율(%)'], ascending=[False, False]).reset_index(drop=True)

tag_all = pd.concat([silence[['appid', 'tag_list']], response[['appid', 'tag_list']]], ignore_index=True)
tag_matrix = build_tag_matrix(tag_all, response[['appid', 'tag_list', 'is_high_satisfaction']])
display(tag_matrix.head(25))

plot_tag_matrix = tag_matrix.head(30).copy()
fig = px.scatter(
    plot_tag_matrix,
    x='유저 반응 확보율(%)',
    y='초기 만족도 high 비율(%)',
    size='total_games',
    text='attribute',
    color='초기 만족도 high 비율(%)',
    color_continuous_scale='Sunsetdark',
    hover_data={'total_games': True, 'response_games': True, 'high_satisfaction_games': True},
    title='태그별 유저 반응 확보율 × 초기 만족도 high 비율 (상위 30개 태그)'
)
fig.add_vline(x=overall_response_capture_rate, line_dash='dash', line_color='gray')
fig.add_hline(y=overall_high_satisfaction_rate, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', width=1200, height=800, coloraxis_showscale=False)
fig.show()

**해석 가이드:** 태그는 장르보다 더 미세한 기획 의도를 보여준다. 우상단 태그는 첫 반응과 만족도 모두 강한 경험 신호이고, 우하단 태그는 관심은 끌지만 구현 난이도나 기대치 관리가 어려웠을 가능성이 있다.

## 6. 종합 정리

이 2x2 매트릭스는 속성을 네 가지 유형으로 나눠 해석하게 해 준다.

1. **우상단:** 반응 확보와 초기 만족도가 모두 강한 속성. 현재 시장에서 비교적 안정적인 포지셔닝 후보다.
2. **우하단:** 반응은 잘 얻지만 만족도가 약한 속성. 기대치 관리 실패나 완성도 문제를 의심할 수 있다.
3. **좌상단:** 반응은 적지만 만족도는 높은 속성. 니치하지만 팬층 만족도가 높은 포지션일 가능성이 있다.
4. **좌하단:** 반응 확보와 만족도 모두 약한 속성. 장르·가격·설명·태그 조합을 함께 재검토할 필요가 있다.

다음 단계로는 이 결과를 바탕으로 `장르 × 가격대`, `장르 × 카테고리` 같은 조합 단위 2x2 분석으로 내려가면 실제 출시 전략 제안까지 연결하기 좋다.